In [2]:
#Importing the NYC dataset that's focused on rodent complaints from 2021-2025
import pandas as pd
import requests #for api route

#Part a. Import
#Reading zip / BBL / additional_details as text: zips and BBLs are IDs, not numbers, and additional_details is ~97% empty, which causeses a mixed-type warning.
#CSV File
df = pd.read_csv(
    "queryed_dataset.csv",
    dtype={"Incident Zip": str, "Additional Details": str, "BBL": str},
    low_memory=False,
)

#Part b. Parse and Transform

#1. Renaming all the long column names, then standardizing all the names to snake_case
df = df.rename(columns={
    "Problem (formerly Complaint Type)": "Complaint Type",
    "Problem Detail (formerly Descriptor)": "Descriptor",
})
df.columns = (df.columns.str.strip().str.lower()
              .str.replace(r"[^\w]+", "_", regex=True).str.strip("_"))

#2. Parsing date strings ("12/31/2025 09:50:33 PM") into datetimes for consistency
date_fmt = "%m/%d/%Y %I:%M:%S %p"
for col in ["created_date", "closed_date", "resolution_action_updated_date"]:
    df[col] = pd.to_datetime(df[col], format=date_fmt, errors="coerce")

#3. Extracting time features into multiple columns
df["year"]        = df["created_date"].dt.year
df["month"]       = df["created_date"].dt.month
df["year_month"]  = df["created_date"].dt.to_period("M")
df["day_of_week"] = df["created_date"].dt.day_name()
df["hour"]        = df["created_date"].dt.hour
df["season"] = df["month"].map({12: "Winter", 1: "Winter", 2: "Winter",
                                3: "Spring", 4: "Spring", 5: "Spring",
                                6: "Summer", 7: "Summer", 8: "Summer",
                                9: "Fall", 10: "Fall", 11: "Fall"})

#4. New feature: days it took to resolve a rodent complaint
df["resolution_days"] = (df["closed_date"] - df["created_date"]).dt.total_seconds() / 86400

#5. Text-to-number conversions
for col in ["x_coordinate_state_plane", "y_coordinate_state_plane"]:   # "1,024,887" -> 1024887
    df[col] = pd.to_numeric(df[col].str.replace(",", ""), errors="coerce").astype("Int64")
df["police_precinct"] = df["police_precinct"].str.extract(r"(\d+)")[0].astype("Int64")  # "Precinct 112" -> 112
df["council_district"] = df["council_district"].astype("Int64")       # 29.0 -> 29 (nullable int)
df["community_board_num"] = df["community_board"].str.extract(r"^(\d+)")[0].astype("Int64")  # "06 QUEENS" -> 6
df["unique_key"] = df["unique_key"].astype(str)                       # an ID, not a quantity

#6. Feature extraction: turn the long resolution sentences into short outcome labels
def outcome(text):
    if pd.isna(text):
        return "No resolution recorded"
    t = text.lower()
    if "earlier complaint" in t:               return "Duplicate"
    if "minor violations" in t:                return "Passed (minor violations)"
    if "found violations" in t or "violations were identified" in t:
                                               return "Violations found"
    if "did not find any violations" in t:     return "Passed"
    if "could not access" in t or "unsafe" in t: return "No access"
    if "referred it to another agency" in t:   return "Referred"
    if "could not locate" in t:                return "Address not found"
    return "Other"

df["inspection_outcome"] = df["resolution_description"].map(outcome)

#7. Convert repeated text values to the category type (to save  memory, and have faster groupby processing)
cat_cols = ["descriptor", "additional_details", "location_type", "address_type", "city",
            "status", "borough", "open_data_channel_type", "day_of_week", "season",
            "inspection_outcome"]
for col in cat_cols:
    df[col] = df[col].astype("category")

print("Number of Complaints and Columns")
print(df.shape)
print(df.dtypes.to_string())


Number of Complaints and Columns
(192716, 53)
unique_key                                object
created_date                      datetime64[ns]
closed_date                       datetime64[ns]
agency                                    object
agency_name                               object
complaint_type                            object
descriptor                              category
additional_details                      category
location_type                           category
incident_zip                              object
incident_address                          object
street_name                               object
cross_street_1                            object
cross_street_2                            object
intersection_street_1                     object
intersection_street_2                     object
address_type                            category
city                                    category
landmark                                  object
facility_type          